# Generate Data (final)

In [17]:
import random
import pandas as pd
from faker import Faker
from datetime import datetime, timedelta

fake = Faker()
Faker.seed(42)
random.seed(42)

def generate_employee_data(num_records=10000):
    employees = []
    cycles = ["Sprint1", "Sprint2", "Sprint3", "Sprint4", "Sprint5"]
    
    for _ in range(num_records):
        first_name = fake.first_name()
        last_name = fake.last_name()
        emp_id = f"E{random.randint(100, 999)}" 
        year, month = random.randint(2023, 2024), random.randint(1, 12)
        month_str = f"{year}-{str(month).zfill(2)}"
        cycle = random.choice(cycles)
        days = random.choice([5, 10])
        
        # Task Metrics
        assigned_tasks = random.randint(5, 10)
        backlog_tasks = random.randint(0, 10)
        total_tasks = assigned_tasks + backlog_tasks
        completed_tasks = random.randint(0, assigned_tasks) 
        task_backlog = total_tasks - completed_tasks
        
        task_completion_rate_assigned = round((completed_tasks / assigned_tasks) * 100, 2) if assigned_tasks > 0 else 0
        task_completion_rate_total = round((completed_tasks / total_tasks) * 100, 2) if total_tasks > 0 else 0
        
        # Task Timing
        sprint_start_date = datetime(year, month, random.randint(1, 25))
        task_start_date = sprint_start_date
        task_due_date = sprint_start_date + timedelta(days=days)
        task_ended = task_due_date if completed_tasks > 0 else None
        
        completion_time_days = (task_ended - task_start_date).days if task_ended else days
        avg_task_completion_time = round(completion_time_days / completed_tasks, 2) if completed_tasks > 0 else 0
        
        # Work Hours & Overtime
        minimum_working_hours = 8
        overtime_frequency = random.randint(0, 3)
        overtime_hours = overtime_frequency * 2
        total_work_hours = (days * minimum_working_hours) + overtime_hours
        
        # Task Priority
        num_high_priority_tasks = random.randint(0, assigned_tasks)
        
        # Leave Metrics
        num_leaves_taken = random.randint(0, 2)
        total_leave_days = num_leaves_taken * random.randint(1, 5)
        total_leave_credits = min(total_leave_days, 5)
        
        employees.append([
            first_name, last_name, emp_id, month_str, cycle, days, total_tasks, assigned_tasks, completed_tasks, 
            task_backlog, task_completion_rate_assigned, task_completion_rate_total, 
            task_start_date.strftime('%Y-%m-%d'), task_ended.strftime('%Y-%m-%d') if task_ended else None, 
            task_due_date.strftime('%Y-%m-%d'), completion_time_days, avg_task_completion_time, 
            minimum_working_hours, overtime_frequency, total_work_hours, num_high_priority_tasks, 
            num_leaves_taken, total_leave_days, total_leave_credits
        ])
    
    columns = [
        "first_name", "last_name", "emp_id", "month", "cycle", "days", "total_tasks", "assigned_tasks", 
        "completed_tasks", "task_backlog", "task_completion_rate_assigned", "task_completion_rate_total", 
        "task_start_date", "task_ended", "task_due_date", "completion_time_days", "avg_task_completion_time", 
        "minimum_working_hours", "overtime_frequency", "total_work_hours", "num_high_priority_tasks", 
        "num_leaves_taken", "total_leave_days", "total_leave_credits"
    ]
    
    return pd.DataFrame(employees, columns=columns)

def label_burnout(row):
    # Burnout thresholds
    high_hours_5_days = 55
    high_hours_10_days = 110
    high_priority_ratio = 0.7
    low_completion_rate = 50

    # Workload-based burnout
    if (row["days"] == 5 and row["total_work_hours"] > high_hours_5_days) or \
       (row["days"] == 10 and row["total_work_hours"] > high_hours_10_days):
        return 1
    
    # Low task completion rate
    if row["task_completion_rate_assigned"] < low_completion_rate:
        return 1
    
    # Too many high-priority tasks
    if row["num_high_priority_tasks"] > row["assigned_tasks"] * high_priority_ratio:
        return 1

    # No leaves taken over multiple sprints
    if row["num_leaves_taken"] == 0:
        return 1

    # Otherwise, not burned out
    return 0

# Generate data
df = generate_employee_data(10000)

# Apply burnout labeling AFTER generating the data
df["burnout"] = df.apply(label_burnout, axis=1)

print(df.head())

df.to_csv("final_synthetic_employee_data.csv", index=False)


  first_name last_name emp_id    month    cycle  days  total_tasks  \
0   Danielle   Johnson   E754  2023-01  Sprint3     5            8   
1     Joshua    Walker   E130  2023-04  Sprint2     5           12   
2       Jill    Rhodes   E928  2023-03  Sprint4    10            9   
3   Patricia    Miller   E199  2024-06  Sprint5    10           12   
4     Robert   Johnson   E949  2024-10  Sprint2     5           15   

   assigned_tasks  completed_tasks  task_backlog  ...  completion_time_days  \
0               6                5             3  ...                     5   
1               9                8             4  ...                     5   
2               7                3             6  ...                    10   
3               5                4             8  ...                    10   
4               5                1            14  ...                     5   

   avg_task_completion_time minimum_working_hours overtime_frequency  \
0                      1.00     

# preprocess

//task_ended= 1278 MISSING VALUES
 employees were assigned tasks but did not complete any, leading to missing (NaN) values in the task_ended column.
//

In [19]:
# check missing values
missing_values = df.isnull().sum()
print("Missing Values:\n", missing_values)

Missing Values:
 first_name                          0
last_name                           0
emp_id                              0
month                               0
cycle                               0
days                                0
total_tasks                         0
assigned_tasks                      0
completed_tasks                     0
task_backlog                        0
task_completion_rate_assigned       0
task_completion_rate_total          0
task_start_date                     0
task_ended                       1278
task_due_date                       0
completion_time_days                0
avg_task_completion_time            0
minimum_working_hours               0
overtime_frequency                  0
total_work_hours                    0
num_high_priority_tasks             0
num_leaves_taken                    0
total_leave_days                    0
total_leave_credits                 0
burnout                             0
dtype: int64


In [20]:
# Check counts of 0 and 1 in the "burnout" column
counts = df['burnout'].value_counts()


print("Counts of burnout values:")
print(counts)

if counts[0] == counts[1]:
    print("The counts of 0 and 1 are equal!")
else:
    print("The counts of 0 and 1 are NOT equal!")


Counts of burnout values:
burnout
1    7630
0    2370
Name: count, dtype: int64
The counts of 0 and 1 are NOT equal!


# train

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

In [25]:
columns_to_exclude = ["first_name", "last_name", "emp_id", "month", "cycle", "task_start_date", "task_due_date", "task_ended"]
df_model = df.drop(columns=columns_to_exclude)


In [26]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df_model[df_model.columns] = scaler.fit_transform(df_model[df_model.columns])


In [ ]:

df_model = pd.get_dummies(df_model)  # One-hot encode categorical variables


X = df_model.drop(columns=["burnout"])  # Features
y = df_model["burnout"]  # Target


In [62]:
# 🔹 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)


In [63]:
# 🔹 Standardize Numeric Features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [64]:
# Convert back to DataFrame to keep feature names
#X_train = pd.DataFrame(X_train, columns=X.columns)
#X_test = pd.DataFrame(X_test, columns=X.columns)

In [65]:
print("Training Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)

Training Data Shape: (8500, 16)
Testing Data Shape: (1500, 16)


In [50]:
# 🔹 Train Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    random_state=42
)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
print("\n🔹 Random Forest Model Performance:")
print("Accuracy:", accuracy_score(y_test, rf_pred))
print(classification_report(y_test, rf_pred))




🔹 Random Forest Model Performance:
Accuracy: 0.971
              precision    recall  f1-score   support

           0       0.96      0.91      0.94       474
           1       0.97      0.99      0.98      1526

    accuracy                           0.97      2000
   macro avg       0.97      0.95      0.96      2000
weighted avg       0.97      0.97      0.97      2000



In [54]:

# 🔹 Train Logistic Regression
log_reg = LogisticRegression(
    C=1.0,
    max_iter=2000,
    class_weight='balanced',
    random_state=42
)
log_reg.fit(X_train, y_train)
log_pred = log_reg.predict(X_test)
print("\n🔹 Logistic Regression Model Performance:")
print("Accuracy:", accuracy_score(y_test, log_pred))
print(classification_report(y_test, log_pred))



🔹 Logistic Regression Model Performance:
Accuracy: 0.8555
              precision    recall  f1-score   support

           0       0.65      0.87      0.74       474
           1       0.95      0.85      0.90      1526

    accuracy                           0.86      2000
   macro avg       0.80      0.86      0.82      2000
weighted avg       0.88      0.86      0.86      2000



In [49]:

# 🔹 Train XGBoost
xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.01,
    max_depth=4,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
print("\n🔹 XGBoost Model Performance:")
print("Accuracy:", accuracy_score(y_test, xgb_pred))
print(classification_report(y_test, xgb_pred))



🔹 XGBoost Model Performance:
Accuracy: 0.9635
              precision    recall  f1-score   support

           0       0.96      0.88      0.92       474
           1       0.96      0.99      0.98      1526

    accuracy                           0.96      2000
   macro avg       0.96      0.94      0.95      2000
weighted avg       0.96      0.96      0.96      2000



In [60]:
# 6️⃣ Show some predictions (First 10)
sample_predictions = pd.DataFrame({
    "Actual": y_test[:10].values,
    "Random Forest": rf_pred[:10],
    "Logistic Regression": log_pred[:10],
    "XGBoost": xgb_pred[:10]
})
print("\n🔍 Sample Predictions:")
print(sample_predictions)


🔍 Sample Predictions:
   Actual  Random Forest  Logistic Regression  XGBoost
0       1              1                    1        1
1       1              1                    1        1
2       0              0                    0        0
3       1              1                    1        1
4       1              1                    1        1
5       1              1                    1        1
6       1              1                    0        1
7       0              0                    0        0
8       1              1                    1        1
9       1              1                    1        1


In [66]:
import joblib

# Save the trained model
joblib.dump(rf_model, r"C:\Users\Sarah\Desktop\try1\model\random_forest_model.pkl")

print("Model saved successfully!")


Model saved successfully!
